# Financial Transaction Fraud Analytics & Risk Investigation

## Project Overview

This project investigates fraud-labelled patterns in a synthetic mobile-money transaction dataset. It combines large-scale exploratory data analysis, statistical association testing, account-behaviour feature engineering, chronological model evaluation, and a Power BI dashboard.

**Important scope note:** This is an analytical case study using synthetic data. The classifier is an experiment, not a reliable or production-ready fraud detection system. The project prioritises transparent analysis and honest model evaluation.

## Objectives

- Assess data quality and transaction-level patterns across 1.72 million records.
- Investigate fraud-label concentration by transaction type, amount, simulation step, and account behaviour.
- Test associations between balance-mismatch indicators and fraud labels.
- Engineer historical sender and recipient behaviour features using prior simulation steps.
- Compare baseline models and evaluate a Random Forest using chronological validation and test sets.
- Communicate findings and limitations through a Power BI dashboard.

## Reproducibility

Place the source CSV at `data/synthetic_mobile_money_transaction_dataset.csv` in the repository root before running the notebook. The dataset is not committed because it is a large source data file. Install dependencies with `pip install -r requirements.txt`.


## 1. Dataset Loading and Initial Inspection

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("data/synthetic_mobile_money_transaction_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Download the synthetic mobile-money CSV and place it in the repository's data/ folder."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)


In [ ]:
# Display all column names
print("COLUMN NAMES:")
print(df.columns.tolist())

# Display first five records
print("\nFIRST 5 RECORDS:")
display(df.head())

# Check data types and non-null counts
print("\nDATASET INFORMATION:")
df.info()

## 2. Data Quality Assessment and Exploratory Data Analysis

In [ ]:

# 1. Missing values
print("MISSING VALUES")
print(df.isnull().sum())

# 2. Duplicate records
print("\nDUPLICATE ROWS")
print(df.duplicated().sum())

# 3. Transaction type distribution
print("\nTRANSACTION TYPE DISTRIBUTION")
print(df['transactionType'].value_counts())

# 4. Fraud distribution
print("\nFRAUD DISTRIBUTION")
print(df['isFraud'].value_counts())

print("\nFRAUD PERCENTAGE")
print(df['isFraud'].value_counts(normalize=True) * 100)

# 5. Numerical summary
print("\nNUMERICAL SUMMARY")
display(df[['step', 'amount',
            'oldBalInitiator', 'newBalInitiator',
            'oldBalRecipient', 'newBalRecipient']].describe())


### Key Findings

- The dataset contains 1,720,181 transactions and 10 columns.
- No missing values or duplicate records were found.
- All fraud-labelled transactions belong to the TRANSFER category.
- Approximately 10.20% of all transactions are labelled as fraudulent.

These findings provide an overview of the dataset and guide the subsequent fraud analysis.

## 3. Fraud Distribution by Transaction Type

In [ ]:

import pandas as pd
import numpy as np

fraud_by_type = (
    df.groupby('transactionType')
      .agg(
          total_transactions=('isFraud', 'count'),
          fraudulent_transactions=('isFraud', 'sum'),
          average_amount=('amount', 'mean')
      )
)

fraud_by_type['fraud_rate_pct'] = (
    fraud_by_type['fraudulent_transactions'] /
    fraud_by_type['total_transactions'] * 100
)

fraud_by_type = fraud_by_type.sort_values(
    'fraud_rate_pct', ascending=False
)

display(fraud_by_type.round(2))


### Key Findings

- TRANSFER is the only transaction type containing fraud-labelled records.
- Out of 569,328 transfer transactions, 175,518 are labelled as fraudulent.
- The fraud-label rate among transfers is approximately 30.83%.
- PAYMENT, DEPOSIT, WITHDRAWAL and DEBIT contain no fraud-labelled records.

**Interpretation:** In this synthetic dataset, fraud labels are concentrated entirely in transfer transactions. This dataset-specific pattern should not be assumed to represent real-world financial fraud.

## 4. Transaction Amount Analysis

In [ ]:

amount_comparison = (
    df.groupby('isFraud')['amount']
      .agg(['count', 'mean', 'median', 'min', 'max'])
)

amount_comparison.index = ['Non-Fraud', 'Fraud']

display(amount_comparison.round(2))



### Key Findings

1. Non-fraud transactions have a higher average transaction amount than fraud-labelled transactions.
2. The median transaction amount is slightly higher for fraud-labelled transactions.
3. Non-fraud transactions show a substantially higher maximum transaction amount.
4. These differences suggest that transaction amount alone may not reliably distinguish fraudulent transactions.


## 5. Balance-Based Feature Engineering

In [ ]:

df['sender_balance_change'] = (
    df['oldBalInitiator'] - df['newBalInitiator']
)

df['recipient_balance_change'] = (
    df['newBalRecipient'] - df['oldBalRecipient']
)

df['sender_balance_difference'] = (
    df['sender_balance_change'] - df['amount']
)

df['recipient_balance_difference'] = (
    df['recipient_balance_change'] - df['amount']
)

display(
    df.groupby('isFraud')[
        ['sender_balance_difference',
         'recipient_balance_difference']
    ].agg(['mean', 'median', 'std']).round(2)
)


In [ ]:

# Filter transfer transactions
transfer_df = df[df['transactionType'] == 'TRANSFER'].copy()

# Compare balance differences by fraud label
transfer_balance_analysis = (
    transfer_df.groupby('isFraud')
    .agg(
        transaction_count=('amount', 'count'),
        avg_amount=('amount', 'mean'),
        sender_difference_mean=('sender_balance_difference', 'mean'),
        sender_difference_std=('sender_balance_difference', 'std'),
        recipient_difference_mean=('recipient_balance_difference', 'mean'),
        recipient_difference_std=('recipient_balance_difference', 'std')
    )
)

display(transfer_balance_analysis.round(2))


In [ ]:

# Calculate balance discrepancies for transfers
transfer_df['sender_abs_difference'] = (
    transfer_df['sender_balance_difference'].abs()
)

transfer_df['recipient_abs_difference'] = (
    transfer_df['recipient_balance_difference'].abs()
)

# Percentage of transfers with balance discrepancies
consistency = (
    transfer_df.groupby('isFraud')
    .agg(
        sender_mismatch_pct=(
            'sender_abs_difference',
            lambda x: (x > 0.01).mean() * 100
        ),
        recipient_mismatch_pct=(
            'recipient_abs_difference',
            lambda x: (x > 0.01).mean() * 100
        )
    )
)

display(consistency.round(2))


# Create transaction-level behavioural features

In [ ]:

# Create transaction-level behavioural features

transfer_df = df[df['transactionType'] == 'TRANSFER'].copy()

transfer_df['sender_balance_change'] = (
    transfer_df['oldBalInitiator'] - transfer_df['newBalInitiator']
)

transfer_df['recipient_balance_change'] = (
    transfer_df['newBalRecipient'] - transfer_df['oldBalRecipient']
)

transfer_df['sender_balance_mismatch'] = (
    transfer_df['sender_balance_change'] - transfer_df['amount']
).abs()

transfer_df['recipient_balance_mismatch'] = (
    transfer_df['recipient_balance_change'] - transfer_df['amount']
).abs()

transfer_df['sender_mismatch_flag'] = (
    transfer_df['sender_balance_mismatch'] > 0.01
).astype(int)

transfer_df['recipient_mismatch_flag'] = (
    transfer_df['recipient_balance_mismatch'] > 0.01
).astype(int)

transfer_df['amount_to_sender_balance_ratio'] = (
    transfer_df['amount'] /
    (transfer_df['oldBalInitiator'].abs() + 1)
)

transfer_df['amount_to_recipient_balance_ratio'] = (
    transfer_df['amount'] /
    (transfer_df['oldBalRecipient'].abs() + 1)
)

display(transfer_df.head())


In [ ]:

# Compare engineered features for fraud and non-fraud transfers

feature_comparison = (
    transfer_df.groupby('isFraud')
    .agg(
        transaction_count=('amount', 'count'),
        median_amount=('amount', 'median'),
        median_sender_ratio=('amount_to_sender_balance_ratio', 'median'),
        median_recipient_ratio=('amount_to_recipient_balance_ratio', 'median'),
        sender_mismatch_rate=('sender_mismatch_flag', 'mean'),
        recipient_mismatch_rate=('recipient_mismatch_flag', 'mean')
    )
)

feature_comparison['sender_mismatch_rate'] *= 100
feature_comparison['recipient_mismatch_rate'] *= 100

display(feature_comparison.round(3))


## 7. Statistical Analysis of Fraud Indicators

In [ ]:

print("Fraud label counts:")
print(transfer_df['isFraud'].value_counts())

print("\nSender mismatch rates:")
print(
    transfer_df.groupby('isFraud')['sender_mismatch_flag']
    .mean().mul(100)
)

print("\nRecipient mismatch rates:")
print(
    transfer_df.groupby('isFraud')['recipient_mismatch_flag']
    .mean().mul(100)
)


In [ ]:

from scipy.stats import chi2_contingency

# Sender mismatch association
sender_table = pd.crosstab(
    transfer_df['isFraud'],
    transfer_df['sender_mismatch_flag']
)

sender_chi2, sender_p, _, _ = chi2_contingency(sender_table)

# Recipient mismatch association
recipient_table = pd.crosstab(
    transfer_df['isFraud'],
    transfer_df['recipient_mismatch_flag']
)

recipient_chi2, recipient_p, _, _ = chi2_contingency(recipient_table)

print("SENDER MISMATCH TEST")
print("Chi-square:", round(sender_chi2, 3))
print("P-value:", sender_p)

print("\nRECIPIENT MISMATCH TEST")
print("Chi-square:", round(recipient_chi2, 3))
print("P-value:", recipient_p)


In [ ]:

import numpy as np
from scipy.stats import chi2_contingency

def cramers_v(data, feature):
    table = pd.crosstab(data['isFraud'], data[feature])
    chi2, p, _, _ = chi2_contingency(table)

    n = table.to_numpy().sum()
    rows, cols = table.shape

    v = np.sqrt(chi2 / (n * min(rows - 1, cols - 1)))

    return {
        'Feature': feature,
        'Chi-square': chi2,
        'P-value': p,
        'Cramers_V': v
    }

results = pd.DataFrame([
    cramers_v(transfer_df, 'sender_mismatch_flag'),
    cramers_v(transfer_df, 'recipient_mismatch_flag')
])

display(results.round(6))


In [ ]:

# Compare transaction amount distributions

amount_stats = (
    transfer_df.groupby('isFraud')['amount']
    .agg(
        count='count',
        mean='mean',
        median='median',
        std='std',
        q75=lambda x: x.quantile(0.75),
        q90=lambda x: x.quantile(0.90),
        q95=lambda x: x.quantile(0.95),
        q99=lambda x: x.quantile(0.99)
    )
)

display(amount_stats.round(2))


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

transfer_df.boxplot(
    column='amount',
    by='isFraud',
    showfliers=False
)

plt.title('Transaction Amount Distribution: Fraud vs Non-Fraud')
plt.suptitle('')
plt.xlabel('Fraud Label (0 = Non-Fraud, 1 = Fraud)')
plt.ylabel('Transaction Amount')
plt.show()


In [ ]:

for label, group in transfer_df.groupby('isFraud'):
    print("\nFraud label:", label)
    print(group['amount'].quantile([0.50, 0.75, 0.90, 0.95, 0.99]))
    print("Mean:", group['amount'].mean())
    print("Standard deviation:", group['amount'].std())


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.boxplot(
    [
        transfer_df.loc[transfer_df['isFraud'] == 0, 'amount'],
        transfer_df.loc[transfer_df['isFraud'] == 1, 'amount']
    ],
    tick_labels=['Non-Fraud', 'Fraud'],
    showfliers=False
)

plt.title('Transaction Amount Distribution: Fraud vs Non-Fraud')
plt.xlabel('Transaction Category')
plt.ylabel('Transaction Amount')
plt.grid(axis='y', alpha=0.3)
plt.show()


In [ ]:

from scipy.stats import mannwhitneyu

non_fraud_amounts = transfer_df.loc[
    transfer_df['isFraud'] == 0, 'amount'
]

fraud_amounts = transfer_df.loc[
    transfer_df['isFraud'] == 1, 'amount'
]

u_stat, p_value = mannwhitneyu(
    fraud_amounts,
    non_fraud_amounts,
    alternative='two-sided'
)

n1 = len(fraud_amounts)
n2 = len(non_fraud_amounts)

rank_biserial = (2 * u_stat / (n1 * n2)) - 1

print("Mann-Whitney U:", u_stat)
print("P-value:", p_value)
print("Rank-biserial correlation:", rank_biserial)


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

transfer_df['log_amount'] = np.log1p(transfer_df['amount'])

plt.figure(figsize=(9, 5))

plt.hist(
    transfer_df.loc[transfer_df['isFraud'] == 0, 'log_amount'],
    bins=50,
    alpha=0.6,
    label='Non-Fraud'
)

plt.hist(
    transfer_df.loc[transfer_df['isFraud'] == 1, 'log_amount'],
    bins=50,
    alpha=0.6,
    label='Fraud'
)

plt.title('Log-Transformed Transaction Amount Distribution')
plt.xlabel('Log(Transaction Amount + 1)')
plt.ylabel('Frequency')
plt.legend()
plt.show()


In [ ]:

plt.figure(figsize=(10, 5))

plt.hist(
    transfer_df.loc[transfer_df['isFraud'] == 0, 'log_amount'],
    bins=60,
    density=True,
    alpha=0.5,
    label='Non-Fraud'
)

plt.hist(
    transfer_df.loc[transfer_df['isFraud'] == 1, 'log_amount'],
    bins=60,
    density=True,
    alpha=0.5,
    label='Fraud'
)

plt.title('Density Comparison of Log Transaction Amounts')
plt.xlabel('Log(Transaction Amount + 1)')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


In [ ]:

# Compare the most frequently occurring transaction amounts

for label in [0, 1]:
    print(f"\nFraud label: {label}")

    print(
        transfer_df.loc[
            transfer_df['isFraud'] == label, 'amount'
        ].value_counts().head(10)
    )


## 8. Fraud Patterns Across Transaction Steps

In [ ]:

# Fraud distribution across transaction steps

step_analysis = (
    transfer_df.groupby('step')
    .agg(
        total_transactions=('isFraud', 'count'),
        fraud_transactions=('isFraud', 'sum')
    )
)

step_analysis['fraud_rate_pct'] = (
    step_analysis['fraud_transactions'] /
    step_analysis['total_transactions'] * 100
)

display(step_analysis.sort_values(
    'fraud_rate_pct', ascending=False
).head(15).round(2))


In [ ]:

# Fraud rate by transaction step range

transfer_df['step_group'] = pd.cut(
    transfer_df['step'],
    bins=[-1, 20, 40, 60, 80, 100, 120, 143],
    labels=[
        '0-20', '21-40', '41-60', '61-80',
        '81-100', '101-120', '121-143'
    ]
)

step_group_analysis = (
    transfer_df.groupby('step_group', observed=False)
    .agg(
        total_transactions=('isFraud', 'count'),
        fraud_transactions=('isFraud', 'sum'),
        fraud_rate=('isFraud', 'mean')
    )
)

step_group_analysis['fraud_rate'] *= 100

display(step_group_analysis.round(2))


In [ ]:

# Calculate fraud rate for every individual step

step_trend = (
    transfer_df.groupby('step')
    .agg(
        total_transactions=('isFraud', 'count'),
        fraud_transactions=('isFraud', 'sum')
    )
)

step_trend['fraud_rate_pct'] = (
    step_trend['fraud_transactions'] /
    step_trend['total_transactions'] * 100
)

plt.figure(figsize=(12, 5))

plt.plot(
    step_trend.index,
    step_trend['fraud_rate_pct'],
    marker='o',
    markersize=3
)

plt.title('Fraud Rate Across Individual Transaction Steps')
plt.xlabel('Transaction Step')
plt.ylabel('Fraud Rate (%)')
plt.grid(alpha=0.3)
plt.show()


## 9. Sender and Recipient Account Fraud Analysis

In [ ]:

# Sender account fraud analysis

sender_analysis = (
    transfer_df.groupby('initiator')
    .agg(
        total_transfers=('isFraud', 'count'),
        fraud_transfers=('isFraud', 'sum')
    )
)

sender_analysis['fraud_rate_pct'] = (
    sender_analysis['fraud_transfers'] /
    sender_analysis['total_transfers'] * 100
)

# Accounts with at least 5 transfers
frequent_senders = sender_analysis[
    sender_analysis['total_transfers'] >= 5
]

print("Total unique sender accounts:", len(sender_analysis))
print("Sender accounts with 5+ transfers:", len(frequent_senders))

print("\nTop 15 sender accounts by fraud count:")
display(
    sender_analysis.sort_values(
        'fraud_transfers', ascending=False
    ).head(15).round(2)
)


In [ ]:

# Overall fraud concentration among sender accounts

sender_analysis['fraud_account_flag'] = (
    sender_analysis['fraud_transfers'] > 0
)

print("Total sender accounts:", len(sender_analysis))

print(
    "Accounts with at least one fraud-labelled transfer:",
    sender_analysis['fraud_account_flag'].sum()
)

print(
    "Percentage of accounts with fraud-labelled transfers:",
    round(sender_analysis['fraud_account_flag'].mean() * 100, 2),
    "%"
)


In [ ]:

# Recipient account fraud analysis

recipient_analysis = (
    transfer_df.groupby('recipient')
    .agg(
        total_transfers=('isFraud', 'count'),
        fraud_transfers=('isFraud', 'sum')
    )
)

recipient_analysis['fraud_rate_pct'] = (
    recipient_analysis['fraud_transfers'] /
    recipient_analysis['total_transfers'] * 100
)

print("Total unique recipient accounts:", len(recipient_analysis))

print("\nTop 15 recipient accounts by fraud count:")

display(
    recipient_analysis.sort_values(
        'fraud_transfers', ascending=False
    ).head(15).round(2)
)


In [ ]:

recipient_analysis['fraud_account_flag'] = (
    recipient_analysis['fraud_transfers'] > 0
)

print(
    "Recipients with at least one fraud-labelled transfer:",
    recipient_analysis['fraud_account_flag'].sum()
)

print(
    "Percentage of recipients with fraud-labelled transfers:",
    round(recipient_analysis['fraud_account_flag'].mean() * 100, 2),
    "%"
)


In [ ]:

# Analyse recipient accounts with at least 5 transfers

frequent_recipients = recipient_analysis[
    recipient_analysis['total_transfers'] >= 5
].copy()

print("Recipient accounts with 5+ transfers:",
      len(frequent_recipients))

print("\nRecipient fraud-rate distribution:")

display(
    frequent_recipients['fraud_rate_pct']
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
    .round(2)
)


In [ ]:

# Group recipients by their observed fraud rate

frequent_recipients['risk_band'] = pd.cut(
    frequent_recipients['fraud_rate_pct'],
    bins=[-0.01, 0, 10, 25, 40, 60, 100],
    labels=[
        'No fraud-labelled transfers',
        'Above 0% to 10%',
        'Above 10% to 25%',
        'Above 25% to 40%',
        'Above 40% to 60%',
        'Above 60%'
    ]
)

risk_distribution = (
    frequent_recipients.groupby(
        'risk_band', observed=False
    )
    .size()
    .reset_index(name='account_count')
)

risk_distribution['account_percentage'] = (
    risk_distribution['account_count'] /
    len(frequent_recipients) * 100
)

display(risk_distribution.round(2))


In [ ]:

# Compare account balance conditions in fraudulent transfers

fraud_pattern_analysis = (
    transfer_df.groupby('isFraud')
    .agg(
        transaction_count=('amount', 'count'),
        sender_zero_balance=(
            'newBalInitiator',
            lambda x: (x == 0).mean() * 100
        ),
        recipient_zero_balance=(
            'newBalRecipient',
            lambda x: (x == 0).mean() * 100
        ),
        sender_negative_balance=(
            'newBalInitiator',
            lambda x: (x < 0).mean() * 100
        ),
        recipient_negative_balance=(
            'newBalRecipient',
            lambda x: (x < 0).mean() * 100
        )
    )
)

display(fraud_pattern_analysis.round(3))



In [ ]:

# Calculate proportion of sender balance consumed

transfer_df['sender_balance_utilization'] = (
    transfer_df['amount'] /
    (transfer_df['oldBalInitiator'].abs() + 1)
)

# Compare utilization between fraud and non-fraud
utilization_analysis = (
    transfer_df.groupby('isFraud')['sender_balance_utilization']
    .agg(
        mean='mean',
        median='median',
        q75=lambda x: x.quantile(0.75),
        q90=lambda x: x.quantile(0.90),
        q95=lambda x: x.quantile(0.95)
    )
)

display(utilization_analysis.round(4))


In [ ]:

# Investigate transactions where amount exceeds
# the sender's absolute starting balance

transfer_df['amount_exceeds_sender_balance'] = (
    transfer_df['amount'] >
    transfer_df['oldBalInitiator'].abs()
)

high_utilization_analysis = (
    transfer_df.groupby('isFraud')
    .agg(
        total_transactions=('amount', 'count'),
        amount_exceeds_balance=(
            'amount_exceeds_sender_balance', 'sum'
        ),
        negative_starting_balance=(
            'oldBalInitiator', lambda x: (x < 0).sum()
        ),
        zero_starting_balance=(
            'oldBalInitiator', lambda x: (x == 0).sum()
        )
    )
)

high_utilization_analysis['exceeds_balance_pct'] = (
    high_utilization_analysis['amount_exceeds_balance'] /
    high_utilization_analysis['total_transactions'] * 100
)

display(high_utilization_analysis.round(2))


In [ ]:

# Display the 10 highest utilization transactions

display(
    transfer_df.nlargest(
        10, 'amount_to_sender_balance_ratio'
    )[
        [
            'amount',
            'oldBalInitiator',
            'newBalInitiator',
            'amount_to_sender_balance_ratio',
            'isFraud'
        ]
    ]
)


In [ ]:

# Recalculate sender balance utilization safely

transfer_df['sender_balance_utilization'] = (
    transfer_df['amount'] /
    transfer_df['oldBalInitiator'].abs().replace(0, np.nan)
)

# Inspect the highest utilization records
extreme_transactions = transfer_df.nlargest(
    10, 'sender_balance_utilization'
)

display(
    extreme_transactions[
        [
            'amount',
            'oldBalInitiator',
            'newBalInitiator',
            'sender_balance_change',
            'sender_balance_utilization',
            'isFraud'
        ]
    ].to_string(index=False)
)


In [ ]:

# Compare actual sender balance reduction with transaction amount

transfer_df['sender_balance_change_check'] = (
    transfer_df['oldBalInitiator'] -
    transfer_df['newBalInitiator']
)

transfer_df['sender_balance_gap'] = (
    transfer_df['sender_balance_change_check'] -
    transfer_df['amount']
)

display(
    transfer_df.groupby('isFraud')['sender_balance_gap']
    .agg(['mean', 'median', 'std'])
    .round(2)
)


In [ ]:

# Compare fraud rates across sender balance conditions

transfer_df['sender_balance_status'] = np.select(
    [
        transfer_df['oldBalInitiator'] < 0,
        transfer_df['oldBalInitiator'] == 0,
        transfer_df['oldBalInitiator'] <= 100,
        transfer_df['oldBalInitiator'] <= 10000
    ],
    [
        'Negative balance',
        'Zero balance',
        'Very low balance',
        'Low balance'
    ],
    default='Normal balance'
)

balance_status_analysis = (
    transfer_df.groupby('sender_balance_status', observed=False)
    .agg(
        total_transactions=('isFraud', 'count'),
        fraud_transactions=('isFraud', 'sum'),
        fraud_rate=('isFraud', 'mean')
    )
)

balance_status_analysis['fraud_rate'] *= 100

display(balance_status_analysis.round(2))



In [ ]:

# Account-level analysis by sender balance category

balance_account_analysis = (
    transfer_df.groupby('sender_balance_status')
    .agg(
        unique_senders=('initiator', 'nunique'),
        total_transactions=('isFraud', 'count'),
        fraud_transactions=('isFraud', 'sum'),
        fraud_rate=('isFraud', 'mean')
    )
)

balance_account_analysis['fraud_rate'] *= 100

display(balance_account_analysis.round(2))


In [ ]:

# Account-level fraud rate within low-balance senders

low_balance_accounts = (
    transfer_df[
        transfer_df['sender_balance_status'] == 'Low balance'
    ]
    .groupby('initiator')
    .agg(
        total_transfers=('isFraud', 'count'),
        fraud_transfers=('isFraud', 'sum')
    )
)

low_balance_accounts['fraud_rate_pct'] = (
    low_balance_accounts['fraud_transfers'] /
    low_balance_accounts['total_transfers'] * 100
)

print("Number of low-balance senders:",
      len(low_balance_accounts))

print("\nAccount-level fraud-rate distribution:")

display(
    low_balance_accounts['fraud_rate_pct']
    .describe()
    .round(2)
)


In [ ]:

# Examine transaction volume for low-balance sender accounts

print("Transaction count distribution:")

display(
    low_balance_accounts['total_transfers']
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
    .round(2)
)

print("\nAccounts with only one transaction:",
      (low_balance_accounts['total_transfers'] == 1).sum())

print("Accounts with 2-5 transactions:",
      low_balance_accounts['total_transfers'].between(2, 5).sum())

print("Accounts with more than 5 transactions:",
      (low_balance_accounts['total_transfers'] > 5).sum())


### Modelling scope and feature-availability check

The exploratory analysis uses transaction balance fields to study post-transaction discrepancies. Some of these fields (including the sender's new balance and derived balance-mismatch features) are only observable after a transaction has been processed. They are useful for retrospective investigation but may not be available to a real-time decision system.

Accordingly, treat model results as an exploratory benchmark. A production-oriented follow-up should rebuild the feature set using only information available before or at authorisation, then repeat the chronological evaluation. Historical account aggregates in this notebook are calculated from earlier simulation steps; this temporal design should be preserved when extending the project.

## 10. Model Preparation and Feature Selection

In [ ]:

# Check the final feature candidates

candidate_features = [
    'step',
    'amount',
    'sender_balance_difference',
    'recipient_balance_difference',
    'sender_balance_mismatch',
    'recipient_balance_mismatch',
    'sender_mismatch_flag',
    'recipient_mismatch_flag',
    'isFraud'
]

model_data = transfer_df[candidate_features].copy()

print("Model dataset shape:", model_data.shape)

print("\nMissing values:")
display(model_data.isnull().sum())

print("\nData types:")
display(model_data.dtypes)


In [ ]:

# Compare feature distributions by fraud label

feature_summary = (
    model_data.groupby('isFraud')
    .mean()
    .T
)

feature_summary.columns = ['Non-Fraud', 'Fraud']

feature_summary['absolute_difference'] = (
    feature_summary['Fraud'] -
    feature_summary['Non-Fraud']
).abs()

display(
    feature_summary.sort_values(
        'absolute_difference',
        ascending=False
    ).round(4)
)


In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt

# Calculate correlations between candidate features
correlation_matrix = model_data.drop(
    columns='isFraud'
).corr()

plt.figure(figsize=(11, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    center=0,
    linewidths=0.5
)

plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:

# Identify feature pairs with absolute correlation above 0.90

corr_pairs = (
    correlation_matrix.where(
        np.triu(
            np.ones(correlation_matrix.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

corr_pairs.columns = [
    'Feature 1', 'Feature 2', 'Correlation'
]

high_corr_pairs = corr_pairs[
    corr_pairs['Correlation'].abs() > 0.90
].sort_values(
    'Correlation',
    key=lambda x: x.abs(),
    ascending=False
)

display(high_corr_pairs.round(4))


In [ ]:

# Check whether sender and recipient balance differences
# are exactly equal for every transaction

print(
    "Differences equal:",
    np.allclose(
        transfer_df['sender_balance_difference'],
        transfer_df['recipient_balance_difference']
    )
)

# Check whether mismatch values are equal
print(
    "Mismatch values equal:",
    np.allclose(
        transfer_df['sender_balance_mismatch'],
        transfer_df['recipient_balance_mismatch']
    )
)

# Check whether mismatch is simply the absolute difference
print(
    "Mismatch is absolute difference:",
    np.allclose(
        transfer_df['sender_balance_mismatch'],
        transfer_df['sender_balance_difference'].abs()
    )
)


In [ ]:

# Calculate the difference between sender and recipient features

balance_gap = (
    transfer_df['sender_balance_difference']
    - transfer_df['recipient_balance_difference']
).abs()

print("Maximum difference:", balance_gap.max())
print("Mean difference:", balance_gap.mean())
print("Median difference:", balance_gap.median())

print(
    "Rows with different values:",
    (balance_gap > 0.01).sum()
)

print(
    "Percentage of rows with different values:",
    round((balance_gap > 0.01).mean() * 100, 2), "%"
)


In [ ]:

print(
    balance_gap.describe(
        percentiles=[0.90, 0.95, 0.99, 0.999]
    )
)

print("\nGap value distribution (rounded to 4 decimals):")

display(
    balance_gap.round(4)
    .value_counts()
    .head(10)
)


In [ ]:

selected_features = [
    'step',
    'amount',
    'sender_balance_difference',
    'sender_mismatch_flag',
    'recipient_mismatch_flag',
    'isFraud'
]

model_data_v2 = transfer_df[selected_features].copy()

print("New dataset shape:", model_data_v2.shape)
display(model_data_v2.head())


In [ ]:

# Compare balance difference patterns between fraud and non-fraud

display(
    model_data_v2.groupby('isFraud')[
        ['sender_balance_difference',
         'sender_mismatch_flag',
         'recipient_mismatch_flag']
    ].agg(['mean', 'median', 'min', 'max']).round(4)
)

# Check how often sender balance difference is approximately zero
model_data_v2['sender_difference_zero'] = (
    model_data_v2['sender_balance_difference'].abs() < 0.01
)

display(
    pd.crosstab(
        model_data_v2['sender_difference_zero'],
        model_data_v2['isFraud'],
        normalize='index'
    ).round(4) * 100
)


In [ ]:

# Fraud rate for each combination of mismatch flags

display(
    model_data_v2.groupby(
        ['sender_mismatch_flag', 'recipient_mismatch_flag']
    )['isFraud']
    .agg(['count', 'sum', 'mean'])
    .assign(
        fraud_rate=lambda x: x['mean'] * 100
    )
    .drop(columns='mean')
    .round(2)
)


In [ ]:

from scipy.stats import chi2_contingency

model_data_v2['both_mismatch'] = (
    (model_data_v2['sender_mismatch_flag'] == 1) &
    (model_data_v2['recipient_mismatch_flag'] == 1)
).astype(int)

contingency_table = pd.crosstab(
    model_data_v2['both_mismatch'],
    model_data_v2['isFraud']
)

chi2, p_value, dof, expected = chi2_contingency(
    contingency_table
)

n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / n)

print("Chi-square:", chi2)
print("P-value:", p_value)
print("Cramer's V:", cramers_v)

display(contingency_table)


In [ ]:

final_candidate_data = transfer_df[
    [
        'step',
        'amount',
        'sender_balance_difference',
        'isFraud'
    ]
].copy()

final_candidate_data['both_mismatch'] = (
    (transfer_df['sender_mismatch_flag'] == 1) &
    (transfer_df['recipient_mismatch_flag'] == 1)
).astype(int)

print("Dataset shape:", final_candidate_data.shape)
display(final_candidate_data.head())


In [ ]:
# Fraud distribution across transaction types

display(
    df.groupby('transactionType')['isFraud']
    .agg(
        total_transactions='count',
        fraud_transactions='sum',
        fraud_rate='mean'
    )
    .assign(
        fraud_rate=lambda x: x['fraud_rate'] * 100
    )
    .round(2)
)

In [ ]:
# Class distribution in our final candidate dataset

class_counts = final_candidate_data['isFraud'].value_counts()

print("Class counts:")
display(class_counts)

print("\nClass percentages:")
display(
    final_candidate_data['isFraud']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

## 11. Train-Test Split and Model Preparation

In [ ]:

from sklearn.model_selection import train_test_split

# Separate predictors and target
X = final_candidate_data.drop(columns='isFraud')
y = final_candidate_data['isFraud']

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting target distribution:")
display(y_test.value_counts(normalize=True).mul(100).round(2))


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create pipeline
logistic_model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train model
logistic_model.fit(X_train, y_train)

print("Logistic Regression training completed!")


## 12. Logistic Regression Model Evaluation

In [ ]:

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    average_precision_score
)

# Generate predictions
y_pred = logistic_model.predict(X_test)
y_prob = logistic_model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("PR-AUC:", round(average_precision_score(y_test, y_prob), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:

# Examine Logistic Regression coefficients

coefficient_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': logistic_model.named_steps[
        'classifier'
    ].coef_[0]
})

coefficient_df['Absolute_Coefficient'] = (
    coefficient_df['Coefficient'].abs()
)

display(
    coefficient_df.sort_values(
        'Absolute_Coefficient',
        ascending=False
    ).round(4)
)


## 13. Random Forest Model Training

In [ ]:

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed!")


In [ ]:

# Predictions
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_prob), 4))
print("PR-AUC:", round(average_precision_score(y_test, rf_prob), 4))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))


In [ ]:

importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    'Importance',
    ascending=False
)

display(importance_df.round(4))

# Visualise feature importance
plt.figure(figsize=(8, 5))

sns.barplot(
    data=importance_df,
    x='Importance',
    y='Feature'
)

plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()


## 14. Time-Based Train-Test Split

In [ ]:

# Sort transfers by simulation step
time_data = final_candidate_data.sort_values(
    'step'
).copy()

# Find the 80th percentile step
split_step = time_data['step'].quantile(0.80)

print("Time split step:", split_step)

# Split chronologically
train_time = time_data[
    time_data['step'] <= split_step
]

test_time = time_data[
    time_data['step'] > split_step
]

print("Training shape:", train_time.shape)
print("Testing shape:", test_time.shape)

print("\nTraining fraud distribution:")
display(train_time['isFraud'].value_counts(normalize=True).mul(100).round(2))

print("\nTesting fraud distribution:")
display(test_time['isFraud'].value_counts(normalize=True).mul(100).round(2))


In [ ]:

# Separate features and target

X_train_time = train_time.drop(columns='isFraud')
y_train_time = train_time['isFraud']

X_test_time = test_time.drop(columns='isFraud')
y_test_time = test_time['isFraud']

# Train Random Forest
rf_time_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_time_model.fit(X_train_time, y_train_time)

print("Time-based Random Forest training completed!")


In [ ]:

# Predictions
y_time_pred = rf_time_model.predict(X_test_time)
y_time_prob = rf_time_model.predict_proba(X_test_time)[:, 1]

print("Accuracy:", round(accuracy_score(y_test_time, y_time_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test_time, y_time_prob), 4))
print("PR-AUC:", round(average_precision_score(y_test_time, y_time_prob), 4))

print("\nClassification Report:")
print(classification_report(y_test_time, y_time_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_time, y_time_pred))


## 15. Account-Level Activity Analysis

In [ ]:

# Account-level activity summary

sender_activity = (
    transfer_df.groupby('initiator')
    .agg(
        total_sent=('amount', 'count'),
        average_sent=('amount', 'mean'),
        total_amount_sent=('amount', 'sum')
    )
)

recipient_activity = (
    transfer_df.groupby('recipient')
    .agg(
        total_received=('amount', 'count'),
        average_received=('amount', 'mean'),
        total_amount_received=('amount', 'sum')
    )
)

print("Unique sender accounts:", sender_activity.shape[0])
print("Unique recipient accounts:", recipient_activity.shape[0])

print("\nSender activity:")
display(sender_activity.describe().round(2))

print("\nRecipient activity:")
display(recipient_activity.describe().round(2))


In [ ]:

# Create historical account behaviour features

def create_historical_features(data, account_col, prefix):

    # Aggregate transactions by account and simulation step
    history = (
        data.groupby([account_col, 'step'])['amount']
        .agg(
            transaction_count='size',
            amount_sum='sum'
        )
        .reset_index()
        .sort_values([account_col, 'step'])
    )

    # Calculate activity from earlier steps only
    history['previous_count'] = (
        history.groupby(account_col)['transaction_count']
        .cumsum() - history['transaction_count']
    )

    history['previous_amount'] = (
        history.groupby(account_col)['amount_sum']
        .cumsum() - history['amount_sum']
    )

    # Calculate historical average amount
    history['previous_average'] = (
        history['previous_amount'] /
        history['previous_count'].replace(0, np.nan)
    )

    # Rename columns
    history = history.rename(columns={
        'previous_count': f'{prefix}_previous_count',
        'previous_average': f'{prefix}_previous_avg_amount'
    })

    return history[
        [
            account_col,
            'step',
            f'{prefix}_previous_count',
            f'{prefix}_previous_avg_amount'
        ]
    ]


# Sender history
sender_history = create_historical_features(
    transfer_df, 'initiator', 'sender'
)

# Recipient history
recipient_history = create_historical_features(
    transfer_df, 'recipient', 'recipient'
)

# Merge historical features into transactions
enhanced_data = transfer_df.merge(
    sender_history,
    on=['initiator', 'step'],
    how='left'
)

enhanced_data = enhanced_data.merge(
    recipient_history,
    on=['recipient', 'step'],
    how='left'
)

# Fill missing history for first-time accounts
history_columns = [
    'sender_previous_count',
    'sender_previous_avg_amount',
    'recipient_previous_count',
    'recipient_previous_avg_amount'
]

enhanced_data[history_columns] = (
    enhanced_data[history_columns].fillna(0)
)

print("Enhanced dataset shape:", enhanced_data.shape)

display(
    enhanced_data[
        ['step', 'initiator', 'recipient'] + history_columns
    ].head(10)
)


In [ ]:

# Check how many transactions have historical account activity

print("Original transfer rows:", len(transfer_df))
print("Enhanced dataset rows:", len(enhanced_data))

print("\nNon-zero historical feature counts:")

display(
    (enhanced_data[history_columns] > 0).sum()
)

print("\nHistorical feature summary:")

display(
    enhanced_data[history_columns].describe().round(2)
)

print("\nExample transactions from later steps:")

display(
    enhanced_data.loc[
        enhanced_data['step'] > 0,
        ['step', 'initiator', 'recipient'] + history_columns
    ].head(10)
)


In [ ]:

print("Original rows:", len(transfer_df))
print("Enhanced rows:", len(enhanced_data))

print("\nNon-zero historical values:")
display((enhanced_data[history_columns] > 0).sum())

print("\nMissing values:")
display(enhanced_data[history_columns].isna().sum())


In [ ]:

# Create improved modelling dataset

improved_features = [
    'step',
    'amount',
    'sender_balance_difference',
    'sender_mismatch_flag',
    'recipient_mismatch_flag',
    'sender_previous_count',
    'sender_previous_avg_amount',
    'recipient_previous_count',
    'recipient_previous_avg_amount',
    'isFraud'
]

improved_data = enhanced_data[improved_features].copy()

# Create combined mismatch feature
improved_data['both_mismatch'] = (
    (improved_data['sender_mismatch_flag'] == 1) &
    (improved_data['recipient_mismatch_flag'] == 1)
).astype(int)

# Remove individual mismatch flags
improved_data.drop(
    columns=[
        'sender_mismatch_flag',
        'recipient_mismatch_flag'
    ],
    inplace=True
)

print("Improved dataset shape:", improved_data.shape)
print("\nMissing values:")
display(improved_data.isnull().sum())

display(improved_data.head())


## 16. Improved Model Preparation with Historical Features

In [ ]:

# Sort data by simulation step
improved_data = improved_data.sort_values('step').copy()

# Split at step 116
train_improved = improved_data[
    improved_data['step'] <= 116
].copy()

test_improved = improved_data[
    improved_data['step'] > 116
].copy()

# Separate features and target
X_train_improved = train_improved.drop(columns='isFraud')
y_train_improved = train_improved['isFraud']

X_test_improved = test_improved.drop(columns='isFraud')
y_test_improved = test_improved['isFraud']

print("Training:", X_train_improved.shape)
print("Testing:", X_test_improved.shape)


In [ ]:

rf_improved = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_improved.fit(
    X_train_improved,
    y_train_improved
)

print("Improved Random Forest training completed!")


In [ ]:

# Generate predictions
y_improved_pred = rf_improved.predict(X_test_improved)

y_improved_prob = rf_improved.predict_proba(
    X_test_improved
)[:, 1]

# Evaluate model
print("Accuracy:",
      round(accuracy_score(y_test_improved, y_improved_pred), 4))

print("ROC-AUC:",
      round(roc_auc_score(y_test_improved, y_improved_prob), 4))

print("PR-AUC:",
      round(average_precision_score(y_test_improved, y_improved_prob), 4))

print("\nClassification Report:")
print(classification_report(y_test_improved, y_improved_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_improved, y_improved_pred))


## 17. Chronological Validation Split

In [ ]:

# Find the chronological validation cutoff
validation_cutoff = train_improved['step'].quantile(0.80)

print("Validation cutoff:", validation_cutoff)

# Split training data chronologically
train_core = train_improved[
    train_improved['step'] <= validation_cutoff
].copy()

validation_data = train_improved[
    train_improved['step'] > validation_cutoff
].copy()

# Separate features and target
X_core = train_core.drop(columns='isFraud')
y_core = train_core['isFraud']

X_val = validation_data.drop(columns='isFraud')
y_val = validation_data['isFraud']

print("\nCore training shape:", X_core.shape)
print("Validation shape:", X_val.shape)

print("\nCore training fraud rate:",
      round(y_core.mean() * 100, 2), "%")

print("Validation fraud rate:",
      round(y_val.mean() * 100, 2), "%")


In [ ]:

rf_tuned = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_core, y_core)

print("Validation model training completed!")


In [ ]:

from sklearn.metrics import precision_recall_curve

val_prob = rf_tuned.predict_proba(X_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_prob
)

print("Validation probabilities generated!")
print("Number of thresholds:", len(thresholds))


In [ ]:

from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in [0.30, 0.40, 0.50]:
    val_pred = (val_prob >= threshold).astype(int)

    threshold_results.append({
        'Threshold': threshold,
        'Precision': precision_score(y_val, val_pred),
        'Recall': recall_score(y_val, val_pred),
        'F1_Score': f1_score(y_val, val_pred),
        'Fraud_Detected': int(
            ((y_val == 1) & (val_pred == 1)).sum()
        ),
        'False_Alarms': int(
            ((y_val == 0) & (val_pred == 1)).sum()
        )
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df.round(4))


In [ ]:

# Calculate F1 score at every available threshold

f1_scores = (
    2 * precision[:-1] * recall[:-1] /
    (precision[:-1] + recall[:-1] + 1e-10)
)

best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]
best_precision = precision[best_index]
best_recall = recall[best_index]
best_f1 = f1_scores[best_index]

print("Best threshold:", round(best_threshold, 4))
print("Precision:", round(best_precision, 4))
print("Recall:", round(best_recall, 4))
print("F1-score:", round(best_f1, 4))


In [ ]:

best_val_pred = (val_prob >= best_threshold).astype(int)

print(
    "Transactions flagged:",
    best_val_pred.sum()
)

print(
    "Percentage flagged:",
    round(best_val_pred.mean() * 100, 2), "%"
)

print(
    "False alarms:",
    ((best_val_pred == 1) & (y_val == 0)).sum()
)


In [ ]:

thresholds_to_check = [
    0.45, 0.50, 0.55, 0.60,
    0.65, 0.70, 0.75, 0.80
]

results = []

for threshold in thresholds_to_check:

    predictions = (val_prob >= threshold).astype(int)

    results.append({
        'Threshold': threshold,
        'Precision': precision_score(y_val, predictions),
        'Recall': recall_score(y_val, predictions),
        'F1_Score': f1_score(y_val, predictions),
        'Fraud_Detected': (
            (predictions == 1) & (y_val == 1)
        ).sum(),
        'False_Alarms': (
            (predictions == 1) & (y_val == 0)
        ).sum(),
        'Flagged_Percentage': predictions.mean() * 100
    })

display(pd.DataFrame(results).round(4))



## 18. Final Model Evaluation on Unseen Test Data

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# Predict probabilities on the final test set
test_prob = rf_tuned.predict_proba(X_test_improved)[:, 1]

# Retain 0.50 as a validation-tested operating threshold.
# The F1-optimal threshold (0.3835) flagged 99.27% of validation
# transactions and generated 62,501 false alarms, so it was not
# selected as a practical operating point.
final_threshold = 0.50
test_predictions = (test_prob >= final_threshold).astype(int)

# Final evaluation
print("Final Threshold:", final_threshold)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_improved, test_predictions))

print("\nClassification Report:")
print(classification_report(y_test_improved, test_predictions))

print("ROC-AUC:", roc_auc_score(y_test_improved, test_prob))
print("PR-AUC:", average_precision_score(y_test_improved, test_prob))

In [ ]:
from pathlib import Path

export_dir = Path('data/processed')
export_dir.mkdir(parents=True, exist_ok=True)

export_path = export_dir / 'fraud_analysis_data.csv'
improved_data.to_csv(export_path, index=False)

print(f"Analysis dataset exported to: {export_path}")

## Final ConclusionA Random Forest classifier was developed to identify fraud-labelled transactionsin a synthetic mobile money transaction dataset.A chronological train-validation-test split was used to evaluate the model.Historical sender and recipient transaction features were engineered usingprevious simulation steps.The final model achieved:- Accuracy: 64%- Precision: 33%- Recall: 22%- F1-score: 0.26- ROC-AUC: 0.5471- PR-AUC: 0.3295The results indicate weak predictive performance. Fraud precision is 33% and recall is 22%, meaning many fraud-labelled transactions are missed. Accuracy (64%) is below the approximately 70.52% majority-class baseline, and ROC-AUC (0.5471) indicates only weak ranking ability.### Threshold selectionA threshold of 0.50 was retained for the final test evaluation after comparingmultiple thresholds on the validation set. Although the F1-optimal thresholdwas 0.3835, it flagged 99.27% of validation transactions and produced 62,501false alarms. The 0.50 threshold was therefore used as a more conservativevalidation-tested operating point. It still has low recall, so the model is notreliable enough for operational fraud detection.### Limitations1. The dataset is synthetic and its fraud labels should not be treated as   representative of real-world fraud.2. All fraud-labelled records in this dataset are TRANSFER transactions.3. The model's performance is only marginally better than random ranking.4. Post-transaction balance information limits its suitability for   real-time fraud prevention.5. The historical activity features use prior simulation steps, but the   transaction-level balance-mismatch fields are retrospective signals.### ConclusionThis project demonstrates an end-to-end machine learning workflow involvingdata exploration, feature engineering, chronological data splitting,classification, and evaluation. The results also highlight the challengesof detecting fraud when the available features have limited predictive power.